# Markov Chains

---
In class today we will be implementing a Markov chain to process sentences

---
## Learning Objectives

1. Students will be able to explain the Markov Chain process
1. Implement a Markov Chain


Markov Chains represent a series of events following the Markov Property: future states are memory-less in that they depend only on the current state. This can be expanded to the idea of variable order Markov models where there is a variable-length memory (eg. 1st order Markov Model). Markov models consist of fully observable states. 

> A common example of this is in predicting the weather: We can clearly see the current weather and would like to predict tomorrow's weather. This is also applicable to biology with one case being CpG islands. 

Our goal today will be to implement a Markov model built from words. For our example text, we will use the classic example of Dr. Seuss because of the repetitive nature of the text.

---
## Train Markov model

For our initial implementation of the Markov Model, we will use the simple example of Dr. Seuss: "One fish two fish red fish blue fish."



In [7]:
import string
from string import punctuation


def build_markov_model(markov_model, new_text):
    """
    <Docstring provided by professor>
    Function to build or add to a 1st order Markov model given a string of text
    We will store the markov model as a dictionary of dictionaries
    The key in the outer dictionary represents the current state
    and the inner dictionary represents the next state with their contents containing
    the transition probabilities.
    Note: This would be easier to read if we were to build a class representation
           of the model rather than a dictionary of dictionaries, but for simplicity
           our implementation will just use this structure.
    Args:
        markov_model (dict of dicts): a dictionary of word:(next_word:frequency pairs)
        new_text (str): a string to build or add to the markov_model
    Returns:
        markov_model (dict of dicts): an updated markov_model

    Pseudocode:
        Take text and change it to something mutable
        Add artificial states for start and end of text
        Building dict of dicts:
            For each word in text:
                Check if the current word is already in the dict, if not, add it
                Check if the next word is already in the dict, if not, add it
                Increment markov_model[word][next_word]
    """

    text = new_text.split() # Turn a string into a list of words

    added_states = ["*S*"] + text + ["*E*"] # add S and E as artificial states to the start and end of the text

    text = added_states  # the variable used in later blocks is "text", so change my updated list back to text
    #print(text) # checking it works

    for i in range(len(text) - 1):
            current_state = text[i]
            next_state = text[i + 1]
            if current_state not in markov_model:   # if current word is not in dictionary
                markov_model[current_state] = {}   # create an empty dict for current word

            # not nested because then the check will fail for repeated words, because they'll already be in current word dict
            if next_state not in markov_model[current_state]:
                markov_model[current_state][next_state] = 1 #initialize next word into dict
            else:
                markov_model[current_state][next_state] += 1

    # Both above and below work as a loop for producing the expected output. Included because I like zip
    """
    # start the loop
    markov_model = {}
    for current_word, next_word in zip(text, text[1:]): #zip here pairs list elements with position
        if current_word not in markov_model:   # if current word is not in dictionary
            markov_model[current_word] = {}   # create an empty dict for current word
        if next_word not in markov_model[current_word]: markov_model[current_word][next_word] = 1 #initialize next word into dict
        else:
            markov_model[current_word][next_word] += 1
    """

    return markov_model

In [8]:
# Given code block
markov_model = dict()
text = "one fish two fish red fish blue fish"
markov_model = build_markov_model(markov_model, text)
print (markov_model)

{'*S*': {'one': 1}, 'one': {'fish': 1}, 'fish': {'two': 1, 'red': 1, 'blue': 1, '*E*': 1}, 'two': {'fish': 1}, 'red': {'fish': 1}, 'blue': {'fish': 1}}


###  Nth order Markov chain
In the above model, each event or word is output from only the previous state with no memory of any prior states. While this is useful in some cases, typical biological applications of Markov chains require higher-order models to accurately capture what we know about a system. For instance, in attempting to identify coding regions of a genome, we know that open reading frames (ORFs) contain codon triplets, and so a third or sixth order Markov chain would better describe these regions. Here you will implement a generalized form of our previous Markov Chain to allow for Nth order chains.


In [9]:
def build_markov_model(markov_model, text, order=1):
    '''
    <Docstring provided by professor>
    Function to build or add to a Nth order Markov model given a string of text
    Args: 
        markov_model (dict of dicts): a dictionary of word:(next_word:frequency pairs)
            or None if a new model is being built
        text (str): a string to build or add to the markov_model
        order (int): the number of previous states to consider for the model
    Returns:
        markov_model (dict of dicts): an updated/new markov_model

    Pseudocode:
    We aren't taking anything from previous function
    Text string still needs to be split into mutable list
    More artificial states need to be added at the start, but the end only needs one.
        Don't hardcode states, it can change: order = n, meaning *S* also = n
    Outer dict needs to be accessible as a tuple
    Increment similar to previous function, but with new order implementation
        check if exists, increment if not
    '''


    new_text = text.split() # Turn a string into a list of words, reusing variables here

    # add S and E as artificial states to the start and end of the text
    added_chars = (["*S*"] * order) + new_text + ["*E*"] # repeat start *S* of text n order times

    # the variable used in later blocks is "text", so change my updated list back to text
    text = added_chars


    for i in range(len(text) - order):
        current = tuple(text[i : i+order]) #current state: start to the increment+order
        next = text[i + order] #next_state
        if current not in markov_model:   # if current word is not in dictionary
            markov_model[current] = {}   # create an empty dict for current word

        # not nested because then the check will fail for repeated words, because they'll already be in current word dict.
        if next not in markov_model[current]:
            markov_model[current][next] = 1 #initialize next word into dict

        else:
            markov_model[current][next] += 1

    return markov_model


In [10]:
# Given code block
markov_model = dict()
text = "one fish two fish red fish blue fish" # Fixed a typo here
markov_model = build_markov_model(markov_model, text, order=2)
markov_model

{('*S*', '*S*'): {'one': 1},
 ('*S*', 'one'): {'fish': 1},
 ('one', 'fish'): {'two': 1},
 ('fish', 'two'): {'fish': 1},
 ('two', 'fish'): {'red': 1},
 ('fish', 'red'): {'fish': 1},
 ('red', 'fish'): {'blue': 1},
 ('fish', 'blue'): {'fish': 1},
 ('blue', 'fish'): {'*E*': 1}}

## Generate text from Markov Model

Markov models are "generative models". That is, the probability states in the model can be used to generate output following the conditional probabilities in the model.

We will now generate a sequence of text from the Markov model. For this section, I recommend using np.random.choice, which allows for you to provide a probability distribution for drawing the next edge in the chain.

In [11]:
import numpy as np

def get_next_word(current_word, markov_model, seed=42):
    '''
    <Docstring provided by professor>
    Function to randomly move a valid next state given a markov model
    and a current state (word)
    Args: 
        current_word (tuple): a word that exists in our model
        markov_model (dict of dicts): a dictionary of word:(next_word:frequency pairs)
    Returns:
        next_word (str): a randomly selected next word based on transition probabilities
    Pseudocode:
        Calculate transition probabilities for all next states from a given state (counts/sum)
        Randomly draw from these to generate the next state

    '''


    nested_dict = markov_model[current_word] # access the nested dictionary from the markov model
    #print(nested_dict) # Checking

    each_word = list(nested_dict.keys()) # pull out the words from the nested dict into a list
    #print(each_word) #Checking


    tally = list(nested_dict.values()) # pull out the tally values from the nested dict into a list
    #print(tally) #checking
    total_of_counts = sum(tally) #Calculate the total sum from the tallies
    #print(total_of_counts) #checking


    probabilities_for_each = [x/total_of_counts for x in tally] # calculate the probabilities for each word
    #print(probabilities_for_each) #checking

    # use numpy to generate the random text based on the probability of each word
    generating_text = np.random.choice(each_word, p=probabilities_for_each)


    return generating_text


def generate_random_text(markov_model, seed=42):
    '''
    <Docstring provided by professor>
    Function to generate text given a markov model
    Args:
        markov_model (dict of dicts): a dictionary of word:(next_word:frequency pairs
    Returns:
        sentence (str): a randomly generated sequence given the model
    Pseudocode:
        Initialize sentence at start state
        Until End State:
            append get_next_word(current_word, markov_model)
        Return sentence
    '''

    np.random.seed(seed) #Seed the model

    # checking the number of start states to determine the order
    s_count = 1 # initialize the count of start states to 1
    s_check = () # initialize empty checker

    while s_check not in markov_model: # while the start state is not a key in the model
        s_check = tuple(['*S*'] * s_count) #multiply by the start state count to get the order
        s_count = s_count + 1 # add to count
    # print(s_check) #checking

    current_word = s_check # the state states are the starting key

    next_word = '' #initialize empty next word

    sentence = [] # initialize empty list

    while next_word != '*E*': # while we have not hit stop condition:
        next_word = get_next_word(current_word, markov_model)# Get next word based on current word from model
        next_state = current_word[1:] + (next_word,) # take last word of current and combine with the next tuple
        current_word = next_state #increment current word to the next current word
        if next_word == '*E*': #We dont want *E* in our sentence
            break
        sentence.append(next_word) # add to the sentence

    text_string = ' '.join(sentence) #join the outputs together cleanly

    return text_string


---

## All the Fish
Up till now, you have only been working with a line or two of the Dr. Seuss' _One Fish, Two Fish_. Now, I want you to build a model using the whole book and try different orders of Markov models.

> **Pro-tip**: Consider how you signify the beginning of the book, beginning of a line, end of a line, and end of the book.

In [12]:
# add more training data to the markov model using data/one_fish_two_fish.txt
# Read whole book

import string
markov_model = dict()

punctuation = str.maketrans('', '', string.punctuation) # hold different punctuation types to remove when file opened

#open text to remove punctuation and newline characters
with open('data/one_fish_two_fish.txt', 'r') as text:
    clean_lines = [line.strip() for line in text.readlines()]
    clean_text = [line.translate(punctuation) for line in clean_lines] # Strip punctuation from each string in the list

    for line in clean_text: #for each line without punctuation
        if line != '': #skip empy lines
            markov_model = build_markov_model(markov_model, line, order=2)

print (generate_random_text(markov_model, seed=7))

Some have two feet and some are blue


---
## Pick Your Poison
There are three texts provided for under `data/`. The first is:
1. Dr. Seuss' "One Fist, Two Fish" (179 lines of text)
2. All of Shakespeare's sonnets (2308 lines of text)
3. Homer's "The Odyssey" (9255 Lines of text)

In [13]:
# An example of a more complex text that we can use to generate more complex output
nth_order_markov_model = dict()

punctuation = str.maketrans('', '', string.punctuation) # hold different punctuation types to remove when file opened
    # Note that this line is redundant to "All the Fish", but if that ever isn't run before this cell, it wouldn't break


with open("data/sonnets.txt", "r") as poison_text: # Open text
    clean_lines = [line.strip() for line in poison_text.readlines()] # Process the lines
    clean_text = [line.translate(punctuation) for line in clean_lines] # Strip punctuation from each string in the list

    poison_markov_model = {} # initialize poison markov model
    for line in clean_text: #for each line without punctuation
        if line != '': # skip empty lines
            poison_markov_model = build_markov_model(poison_markov_model, line, order=2)

print (generate_random_text(poison_markov_model,seed=7))

But heaven in thy affairs fall by thy picture or my love thou my rose in it
